In [1]:
from ovo import db, storage, schedulers, descriptors, design_logic, descriptor_logic, \
    models_proteinqc, project_logic, descriptors_bindcraft
from ovo.app.components import molstar_notebook, StructureVisualization
import os
import seaborn as sns
from matplotlib import pyplot as plt

%config InlineBackend.figure_format = 'retina'

Registering plugin ovo_promb
Registering plugin ovo_proteindj


OVO home /home/username/ovo

In [2]:
project, project_round = project_logic.get_or_create_project_round("OVO Publication Examples 1", "Binder design")

In [3]:
pools = db.Pool.select(round_id=project_round.id)
for pool in pools:
    print(pool.id, pool.name)

rsj 4ZXB BindCraft default
avz 4ZXB 1000 designs default weights
mmo 4ZXB 1000 designs beta sheet
qki Top designs diversification


In [4]:
# Pool id from previous notebooks
POOL_IDS = ['rsj']

In [5]:
designs = db.Design.select(pool_id__in=POOL_IDS, accepted=True)
len(designs)

13

In [6]:
workflow = models_proteinqc.ProteinQCWorkflow(
    chains=['B'], # NOTE that Bindcraft stores design chain as B - this might change in the future!
    design_ids=[design.id for design in designs],
    tools=['seq_composition', 'esm_1v', 'esm_if', 'dssp', 'peppatch', 'proteinsol'],
)
workflow.validate()
workflow.get_table_row()

Workflow  type    ProteinQC workflow
dtype: object

In [7]:
print(schedulers.keys())

SCHEDULER_KEY = 'slurm_singularity'

dict_keys(['slurm_singularity', 'pbs_singularity', 'local_singularity', 'local_conda', 'local_single_gpu'])


In [8]:
#
# SUBMIT - note that running this multiple times will submit a the workflow each time!
#
descriptor_job = descriptor_logic.submit_descriptor_workflow(
    workflow=workflow,
    scheduler_key=SCHEDULER_KEY,
    project_id=project.id
)
descriptor_job.id

Submitting workflow: nextflow run -with-trace trace.txt -work-dir /home/username/ovo/workdir/work /home/username/projects/ovo-open-source/ovo/ovo/pipelines/proteinqc --publish_dir output --reference_files_dir /home/username/ovo/reference_files --shared_modules ovo:/home/username/projects/ovo-open-source/ovo/ovo,ovo_promb:/home/username/projects/ovo-open-source/ovo-promb/ovo_promb,ovo_proteindj:/home/username/projects/ovo-proteindj/ovo_proteindj -config /home/username/projects/ovo-open-source/ovo/ovo/pipelines/nextflow_default.config -config /home/username/projects/ovo-open-source/ovo/ovo/pipelines/proteinqc/nextflow.config -profile singularity -config /home/username/ovo/nextflow_slurm_singularity.config -ansi-log false -bg --input_pdb /home/username/ovo/workdir/inputs/04/b561a167d1bd9d79fb8edd60bbe0d22ddb8f86/input_pdb_paths.txt --tools seq_composition,esm_1v,esm_if,dssp,peppatch,proteinsol --chains B --batch_size 50
Execution directory: /home/username/ovo/workdir/execdir/0eaf52f0-c169

'000a4b'

In [9]:
descriptor_logic.process_results(descriptor_job)

Waiting for job 0eaf52f0-c169-11f0-b1d0-0248d8152725 to finish...


In [10]:
db.DescriptorValue.count(design_id__in=db.Design.select_values('id', pool_id__in=POOL_IDS, accepted=True))

1714

In [11]:
values = descriptor_logic.get_wide_descriptor_table(
    design_ids=db.Design.select_values('id', pool_id__in=POOL_IDS, accepted=True),
)
print(len(values), 'designs')
print(len(values.columns), 'descriptors')
values.head()

13 designs
133 descriptors


,Sequence B,MPNN score,MPNN seq recovery,AF2 pLDDT,AF2 pTM score,AF2 Interface pTM score,AF2 pAE,AF2 Interface PAE,AF2 Interface pLDDT,AF2 Secondary Structure pLDDT,...,Normalized positive top1 patch area at pH 7.4,Negative patch area at pH 7.4,Normalized negative patch area at pH 7.4,Negative top1 patch area at pH 7.4,Normalized negative top1 patch area at pH 7.4,Electrostatic volume integral at pH 7.4,Positive electrostatic regions volume integral at pH 7.4,Negative electrostatic regions volume integral at pH 7.4,Positive electrostatic volume integral at pH 7.4,Negative electrostatic volume integral at pH 7.4
design_id,,,,,,,,,,,,,,,,,,,,,
ovo_rsj_rank01_bindcraft,MREDPEFVEVMEELAKVIEKYAPKATSEPIPKSVRDENGKVNTSEV...,0.87,0.63,89.0,0.90,0.83,4.96,5.27,86.0,93.0,...,0.006360,1857.575613,0.330403,1736.474753,0.308863,-36524.320846,1148.296154,-25831.176939,3399.649156,-39923.970002
ovo_rsj_rank02_bindcraft,MKPTQADVDKLIDEMMSLQDHPEGPTWEEFSRVMRNIWEYMENGDL...,0.96,0.49,94.0,0.90,0.83,3.72,4.65,93.0,94.0,...,0.007897,1489.507855,0.339800,905.230772,0.206509,-31984.708596,1266.263048,-22859.997680,3288.490000,-35273.198596
ovo_rsj_rank03_bindcraft,SMKKKELEKEVDKILEEMVKRAEEVSRNGTPEEGDKYWKEEMEKLM...,1.07,0.29,92.0,0.90,0.83,4.34,4.65,93.0,93.0,...,0.039639,819.670088,0.182835,410.877210,0.091650,-16045.718668,2693.580476,-11465.700395,5933.390554,-21979.109222
ovo_rsj_rank04_bindcraft,SPELEKLKKEVDEILKEMVERAEEVSRNGTPEEGDKYWKEEMEKLM...,1.08,0.29,92.0,0.90,0.82,4.34,4.96,91.0,92.0,...,0.005562,1157.038447,0.262139,970.704469,0.219923,-29496.396802,383.832596,-16679.919238,912.114584,-30408.511387
ovo_rsj_rank05_bindcraft,MLSEELKEKIKELLEEGYNLVKEMVSKGKPNFKEIEEKLKPKFEEI...,1.05,0.30,89.0,0.89,0.82,5.27,6.20,91.0,91.0,...,0.001419,2726.446233,0.434953,2654.727714,0.423512,-55180.545034,577.224616,-39145.834783,1813.723207,-56994.268240


In [12]:
print(values.columns.tolist())

['Sequence B', 'MPNN score', 'MPNN seq recovery', 'AF2 pLDDT', 'AF2 pTM score', 'AF2 Interface pTM score', 'AF2 pAE', 'AF2 Interface PAE', 'AF2 Interface pLDDT', 'AF2 Secondary Structure pLDDT', 'Unrelaxed Clashes', 'Relaxed Clashes', 'PyRosetta Binder Energy Score', 'Surface Hydrophobicity', 'Shape Complementarity', 'PackStat', 'PyRosetta Interface dG', 'Delta SASA', 'PyRosetta dG/dSASA', 'Interface SASA %', 'Interface Hydrophobicity', 'Num Interface Residues', 'Num Interface Hbonds', 'Interface Hbonds %', 'N Interface Unsat Hbonds', 'Interface Unsat Hbonds %', 'Interface Helix %', 'Interface Beta Sheet %', 'Interface Loop %', 'Binder Helix %', 'Binder BetaSheet %', 'Binder Loop %', 'AF2 Target-aligned Binder RMSD', 'AF2 Target RMSD', 'AF2 Binder pLDDT', 'AF2 Binder pTM score', 'AF2 Binder pAE', 'AF2 Binder-aligned Binder RMSD', 'Binder Interface Residues', 'Sequence length', 'Ala %', 'Cys %', 'Asp %', 'Glu %', 'Phe %', 'Gly %', 'His %', 'Ile %', 'Lys %', 'Leu %', 'Met %', 'Asn %', 'P

In [13]:
values.to_csv('../../data/results/05_bindcraft_binder_design/descriptors.csv')

### Save descriptors for all designs including not accepted

In [14]:
all_values = descriptor_logic.get_wide_descriptor_table(
    pool_ids=POOL_IDS,
    descriptor_keys=[d.key for d in descriptors_bindcraft.DESCRIPTORS]
)
print(len(all_values), 'designs')
print(len(all_values.columns), 'descriptors')
all_values.head()

108 designs
39 descriptors


,Sequence B,MPNN score,MPNN seq recovery,AF2 pLDDT,AF2 pTM score,AF2 Interface pTM score,AF2 pAE,AF2 Interface PAE,AF2 Interface pLDDT,AF2 Secondary Structure pLDDT,...,Binder Helix %,Binder BetaSheet %,Binder Loop %,AF2 Target-aligned Binder RMSD,AF2 Target RMSD,AF2 Binder pLDDT,AF2 Binder pTM score,AF2 Binder pAE,AF2 Binder-aligned Binder RMSD,Binder Interface Residues
design_id,,,,,,,,,,,,,,,,,,,,,
ovo_rsj_rejected01_bindcraft,NWWEMSGYEYAKEMVKNLSEFGYDLVEEMKKRGYTPEAIKKMEEAV...,1.02,0.34,89.0,0.91,0.88,5.27,5.27,91.0,91.0,...,80.00,0.0,20.00,1.89,0.22,90.0,0.81,4.65,2.12,"B2,B10,B13,B14,B17,B18,B20,B21,B24,B25,B28,B29..."
ovo_rsj_rejected02_bindcraft,DWWKMSGYEYAKEMVENLSEFGYDLVREMRERGYTKEAIERMEKAV...,1.02,0.37,93.0,0.91,0.88,4.34,4.65,92.0,94.0,...,80.00,0.0,20.00,1.36,0.24,93.0,0.82,3.72,1.55,"B2,B3,B7,B10,B11,B13,B14,B17,B18,B20,B21,B24,B..."
ovo_rsj_rejected03_bindcraft,SRLDEELAEMNAEWDKLSSKMTFVPWTDMDPYEAESDALWDAFMAA...,1.13,0.38,91.0,0.91,0.88,4.34,4.65,86.0,94.0,...,74.07,0.0,25.93,1.28,0.22,86.0,0.72,5.27,2.05,"B7,B10,B11,B14,B15,B18,B21,B22,B23,B24,B25,B26..."
ovo_rsj_rejected04_bindcraft,SAMDEELAKMNAEWDKLSSKMTFKPWDDMDPYEAESDALWDKFMAA...,1.12,0.38,90.0,0.92,0.88,4.34,4.65,86.0,94.0,...,74.69,0.0,25.31,1.31,0.22,85.0,0.69,5.27,2.30,"B7,B10,B11,B14,B18,B19,B21,B22,B23,B24,B25,B26..."
ovo_rsj_rejected05_bindcraft,SMIDEELAQMNKEWDKLSSKMTFVPWEDMDKYEEESDKLWDRFMEA...,1.11,0.36,90.0,0.91,0.87,4.65,4.65,88.0,91.0,...,79.01,0.0,20.99,1.62,0.23,85.0,0.69,5.27,1.73,"B7,B10,B11,B14,B18,B19,B21,B22,B23,B24,B26,B29..."


In [15]:
all_values.to_csv('../../data/results/05_bindcraft_binder_design/seq_descriptors_all_designs.csv')